In [ ]:
import sys; sys.path.append('../../../'); sys.path.append('../../')
import inflation, numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation
from numpy.linalg import norm
import MeshFEM, parallelism, benchmark, utils
import periodic_unit_helper
import numpy.linalg as la

In [ ]:
n_vx = [[0, 0], [0, 1], [0, 2],
        [1, 0], [1, 1], [1, 2],
        [2, 0], [2, 1], [2, 2]]
n_edge = [(0, 1), (1, 2), 
          (3, 4), (4, 5),
          (6, 7), (7, 8),
          (0, 3), (3, 6),
          (1, 4), (4, 7),
          (2, 5), (5, 8)]
triArea = 0.5

In [ ]:
m, fuseMarkers, fuseSegments = wall_generation.triangulate_channel_walls(n_vx, n_edge, triArea, flags="Y")

In [ ]:
fuseMarkers = [0] * 9

In [ ]:
fuseMarkers[4] = 1
for i in range(3):
    fuseMarkers[i] = 1
    fuseMarkers[i + 6] = 1

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, np.array(fuseMarkers) == 1, epsilon = 1e-5)

In [ ]:
fixedVars = periodic_unit_helper.get_center_fixedVars(ipu)

In [ ]:
import py_newton_optimizer
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
import time, vis
# When doing gradient validation, need to disable tension field theory
ipu.sheet.setUseTensionFieldEnergy(False)
ipu.sheet.setUseHessianProjectedEnergy(False)
ipu.sheet.pressure = 1
opts.niter = 200
framerate = 5 # Update every 5 iterations


In [ ]:
ipu.numVars()

In [ ]:
ipu.setVars([1., 1., 0., 0., 0., -1., 0., 0., 1., 0., 0., 1., 0., 0., -1., 0., 0., 1., 0, 0])

In [ ]:
viewer.update()

In [ ]:
import fd_validation

In [ ]:
fd_validation.gradConvergencePlot(ipu.sheet, customArgs = {"energyType": inflation.InflatableSheet.EnergyType.Full})

In [ ]:
fd_validation.hessConvergencePlot(ipu.sheet, customArgs = {"energyType": inflation.InflatableSheet.EnergyType.Full})

In [ ]:
fd_validation.gradConvergencePlot(ipu, customArgs = {"energyType": inflation.InflatablePeriodicUnit.EnergyType.Full})

In [ ]:
fd_validation.hessConvergencePlot(ipu, customArgs = {"energyType": inflation.InflatablePeriodicUnit.EnergyType.Full})

In [ ]:
ipu.sheet.setUseTensionFieldEnergy(True)
ipu.sheet.setUseHessianProjectedEnergy(False)
ipu.sheet.disableFusedRegionTensionFieldTheory(False)

ipu.sheet.pressure = 3

In [ ]:
fixedVars, hessianShift = [], 1e-6

In [ ]:
benchmark.reset()

opts.niter = 10
framerate = 1 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)
benchmark.report()

In [ ]:
ipu.sheet.setUseTensionFieldEnergy(False)

In [ ]:
fd_validation.gradConvergencePlot(ipu, customArgs = {"energyType": inflation.InflatablePeriodicUnit.EnergyType.Full})

In [ ]:
fd_validation.hessConvergencePlot(ipu, customArgs = {"energyType": inflation.InflatablePeriodicUnit.EnergyType.Full})